# Label batch 2: candidates spread across distance bands

Batch 1 only sampled candidates within 60 km, so the labels carried no distance signal and Ridge learned the wrong
distance sign. This notebook draws new candidates for all 9 services (28 queries, about 400 rows), stratified by distance band (including
facilities beyond the ranking radius), and writes `labels_batch2_todo.csv` for labeling.

Labelers fill **evidence_grade** only (0-3, judged from the quotes). The distance policy is applied in code
(`train_ridge.distance_penalty`), identically for batch 1 and batch 2.

Run from the repository root of the Databricks Git folder.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
import ranking
import train_ridge as tr

SERVICES = ["dialysis", "ct_scan", "mri", "cardiac_surgery", "c_section", "nicu", "icu",
            "emergency_trauma", "burn_care"]
# Cities in priority order; ones the dataset does not know are skipped and the next one is used
CITIES = ["Lucknow", "Ahmedabad", "Jaipur", "Kolkata", "Patna", "Pune", "Bhopal", "Chennai", "Hyderabad",
          "Nagpur", "Kochi", "Indore", "Mumbai", "Delhi", "Bengaluru", "Surat", "Chandigarh", "Coimbatore",
          "Visakhapatnam", "Guwahati", "Bhubaneswar", "Ludhiana", "Varanasi", "Kanpur", "Vadodara", "Madurai",
          "Raipur", "Ranchi", "Dehradun", "Thiruvananthapuram", "Mysore", "Bangalore", "New Delhi", "Agra",
          "Nashik", "Rajkot", "Amritsar", "Jodhpur", "Mangalore", "Vijayawada"]
N_QUERIES = 28   # 28 queries x 3 bands x 5 = up to 420 rows (~100 per labeler)
PER_BAND = 5
SEED = 42

In [ ]:
fac = spark.table("workspace.datathon.facilities_clean").toPandas()
evi = spark.table("workspace.datathon.facility_service_evidence").toPandas()

fac["unique_id"] = fac["unique_id"].astype(str).str.strip()
evi["unique_id"] = evi["unique_id"].astype(str).str.strip()
flag = lambda c, default: fac[c].fillna(default).astype(bool)
usable = fac[~fac["unique_id"].duplicated(keep=False)
             & ~flag("flag_row_corrupted", True)
             & ~flag("flag_coords_outside_india", False)
             & fac["latitude"].notna() & fac["longitude"].notna()].copy()
print(f"{len(usable)} usable facilities, {len(evi)} evidence rows")

In [ ]:
def city_centre(city):
    m = usable[(usable["location_confidence"] == "high")
               & (usable["city_clean"].astype(str).str.strip().str.lower() == city.lower())]
    return (float(m["latitude"].median()), float(m["longitude"].median())) if len(m) >= 5 else None


def bands(service):
    return [(0, 10), (10, 25), (25, 60)] if service in ranking.URGENT_SERVICES else [(0, 20), (20, 50), (50, 100)]


def pick(df, n, rng):
    """Random sample that rotates across evidence levels so each band mixes strong, moderate and weak."""
    df = df.sample(frac=1, random_state=int(rng.integers(1_000_000_000)))
    groups = [g for _, g in df.groupby("evidence_level")]
    out, i = [], 0
    while len(out) < n and any(len(g) > i for g in groups):
        for g in groups:
            if len(g) > i and len(out) < n:
                out.append(g.iloc[i])
        i += 1
    return pd.DataFrame(out)


def quote_text(quotes, n=3):
    items = list(quotes) if quotes is not None else []
    return " | ".join(f"{q['field'].replace('_list', '')}: {q['sentence']}" for q in items[:n])


rng = np.random.default_rng(SEED)
rows = []
qi = 0
for city in CITIES:
    if qi >= N_QUERIES:
        break
    service = SERVICES[qi % len(SERVICES)]
    centre = city_centre(city)
    if centre is None:
        print(f"SKIP {city}: city not found, trying the next city")
        continue
    qi += 1
    cand = evi[evi["service"] == service].drop_duplicates("unique_id").merge(usable, on="unique_id")
    cand["approx_road_km"] = tr.haversine_km(centre[0], centre[1], cand["latitude"], cand["longitude"]) * tr.ROAD_FACTOR
    for lo, hi in bands(service):
        chosen = pick(cand[(cand["approx_road_km"] >= lo) & (cand["approx_road_km"] < hi)], PER_BAND, rng)
        for _, c in chosen.iterrows():
            rows.append({
                "query_id": f"B2Q{qi:02d}", "service_code": service, "query_city": city,
                "query_lat": centre[0], "query_lon": centre[1], "facility_id": c["unique_id"],
                "facility_name": c["name"], "facility_type": c["facility_type"], "facility_city": c["city_clean"],
                "approx_road_km": round(float(c["approx_road_km"]), 1), "distance_band": f"{lo}-{hi} km",
                "location_confidence": c["location_confidence"], "rule_evidence_level": c["evidence_level"],
                "quotes": quote_text(c["quotes"]), "quality_notes": c.get("quality_notes"),
            })

todo = pd.DataFrame(rows).sort_values(["query_id", "approx_road_km"]).reset_index(drop=True)
todo.insert(0, "row_id", [f"B2R{i:03d}" for i in range(1, len(todo) + 1)])
todo["evidence_grade"] = ""   # labelers fill 0-3
todo["labeler"] = ""
todo["note"] = ""
print(todo.groupby(["query_id", "service_code", "query_city", "distance_band"]).size().to_string())
print(f"\n{todo['query_id'].nunique()} queries, {len(todo)} rows to label")

In [ ]:
out = REPO_ROOT / "labels_batch2_todo.csv"
todo.to_csv(out, index=False)
print(f"Wrote {out}. Download it from the Workspace file browser, or use the download button below.")
display(todo)